# Inventory Optimization for Amazon FBA Brands
### FlexCore Portfolio Analysis — Python · Pandas · Plotly

---

**Brand:** FlexCore — fitness accessories sold exclusively on Amazon Europe (ES, IT, DE)  
**Data:** 8 SKUs | 90 days of simulated demand history | Seasonality: New Year + Spring peaks  
**Stack:** Python · Pandas · NumPy · Plotly · Streamlit  

This notebook walks through the full inventory management workflow for an Amazon-first brand:
from raw demand data to actionable decisions on reordering, campaign timing, and IPI health.

> **Portfolio note:** FlexCore is a fictitious brand. The model applies 1:1 to any Amazon FBA brand — replace the input CSVs with your own data (see Section 7).

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

from data.simulate_data import generate_data
from src.abc_analysis import run_abc_analysis, plot_pareto
from src.inventory_model import compute_metrics, STATUS_COLOR
from src.ipi_simulator import estimate_ipi, ipi_label, ipi_advice, plot_gauge
from src.campaign_checker import check_readiness

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

sku_df, demand_df = generate_data(seed=42)
print(f"Loaded: {len(sku_df)} SKUs | {len(demand_df):,} demand records")
print(f"Period: {demand_df['date'].min()} → {demand_df['date'].max()}")

---
## 1. The Problem: Over-stock vs. Under-stock

Amazon FBA brands live with a structural tension: holding too much stock wastes capital and triggers Amazon penalties;
holding too little loses sales and damages your algorithmic ranking.
The goal of inventory management is not to minimize stock — it's to minimize the **cost of being wrong** in either direction.

In [ ]:
tension = pd.DataFrame([
    {
        'Scenario': 'Over-stock',
        'Capital impact': 'Cash locked in FBA warehouse',
        'Amazon fee impact': 'Higher monthly storage fees; long-term fees after 365 days',
        'IPI impact': 'Excess inventory flag (>90 days of supply) penalizes score',
        'Model signal': 'days_of_supply > 90 → EXCESS',
    },
    {
        'Scenario': 'Under-stock',
        'Capital impact': 'Lost sales; competitor takes Buy Box',
        'Amazon fee impact': 'No storage fees, but organic ranking degrades',
        'IPI impact': 'In-stock rate drops → IPI penalty; storage limits tighten',
        'Model signal': 'current_stock < safety_stock → CRITICAL',
    },
]).set_index('Scenario')

tension

---
## 2. The Data: FlexCore Portfolio

FlexCore sources 8 SKUs from manufacturers in China and Vietnam, with lead times of **25–45 days**.
Products ship directly to Amazon FBA warehouses (Pan-European FBA).

Demand has two seasonal peaks:
- **January:** +60% above baseline (New Year fitness resolutions)
- **March–April:** +30% (spring outdoor season)

The current stock snapshot was engineered to include all realistic operational states:
excess inventory, near-stockout situations, and a stranded listing.

In [ ]:
sku_display = sku_df.rename(columns={
    'sku': 'SKU', 'product_name': 'Product', 'price': 'Price (€)',
    'cogs': 'COGS (€)', 'lead_time_days': 'Lead Time (d)',
    'current_stock': 'Current Stock', 'is_stranded': 'Stranded'
}).set_index('SKU')

sku_display.style.format({'Price (€)': '€{:.2f}', 'COGS (€)': '€{:.2f}'})\
    .applymap(lambda v: 'color: #e74c3c; font-weight: bold' if v is True else '', subset=['Stranded'])

In [ ]:
demand_df['date'] = pd.to_datetime(demand_df['date'])
daily = demand_df.merge(sku_df[['sku', 'product_name']], on='sku')

fig = px.line(
    daily, x='date', y='units_sold', color='product_name',
    title='Daily Demand by SKU — FlexCore Portfolio (90 days)',
    labels={'units_sold': 'Units sold/day', 'date': '', 'product_name': 'SKU'},
    height=420
)
fig.update_layout(
    legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0),
    plot_bgcolor='white'
)
fig.show()

---
## 3. ABC Analysis: Where Is the Revenue?

**Pareto principle applied to inventory:** a small number of SKUs generate most of the revenue.
Treating all SKUs equally wastes operational bandwidth on low-impact products.

ABC classification by 90-day revenue contribution:

| Class | Revenue threshold | Inventory strategy |
|---|---|---|
| **A** | Top 80% of revenue | Highest service level (99%), tighter reorder cycles, never stockout |
| **B** | Next 15% (80–95%) | Standard service level (95–98%) |
| **C** | Bottom 5% | Minimal safety stock; liquidate if excess |

The Pareto chart below shows each SKU’s revenue share (bars) and the cumulative curve (line).

In [ ]:
abc_df = run_abc_analysis(sku_df, demand_df)

abc_display = abc_df[['sku', 'product_name', 'revenue_90d', 'revenue_pct', 'cumulative_pct', 'abc_class']].copy()
abc_display.columns = ['SKU', 'Product', 'Revenue 90d (€)', 'Share (%)', 'Cumulative (%)', 'Class']
abc_display = abc_display.set_index('SKU')

class_colors = {'A': 'background-color: #d5f5e3', 'B': 'background-color: #fef9e7', 'C': 'background-color: #fadbd8'}

abc_display.style\
    .format({'Revenue 90d (€)': '€{:,.0f}', 'Share (%)': '{:.1f}%', 'Cumulative (%)': '{:.1f}%'})\
    .applymap(lambda v: class_colors.get(v, ''), subset=['Class'])

In [ ]:
fig = plot_pareto(abc_df)
fig.show()

**Key finding:** Class A (FC-002 Yoga Mat + FC-001 Resistance Bands) concentrates ~80% of revenue
despite being just 2 of 8 SKUs. A stockout on either would be disproportionately costly.

FC-002 is currently EXCESS (→ IPI drag). The tension: our highest-revenue SKU has too much stock.
The right action is not to stop selling — it’s to run a sell-through campaign to reduce the excess
without stockout risk.

---
## 4. Safety Stock & Reorder Point

**Safety stock (SS)** is the statistical buffer that protects against demand variability and lead time uncertainty.
It answers the question: *how much extra stock do I need to avoid a stockout with X% probability?*

We use the **standard deviation method** — the industry standard for replenishment planning:

$$SS = Z \times \sigma_{demand} \times \sqrt{lead\_time}$$

| Parameter | Meaning | FlexCore values |
|---|---|---|
| **Z** | Service level factor | 1.65 (95%) / 2.05 (98%) / 2.58 (99%) |
| **σ_demand** | Std dev of daily demand | Calculated from 90-day history |
| **√lead_time** | Amplifies uncertainty over longer supply chains | 5.0–6.7 for 25–45 day LTs |

**Reorder Point (ROP):** the stock level that triggers a new purchase order to the supplier.

$$ROP = \mu_{demand} \times lead\_time + SS$$

When stock falls to ROP, you’ve consumed enough that the next shipment must arrive before you run out.

In [ ]:
inv_df = compute_metrics(sku_df, demand_df, service_level='95%')

cols = ['sku', 'product_name', 'avg_daily_demand', 'safety_stock',
        'reorder_point', 'current_stock', 'days_of_supply', 'status']
inv_display = inv_df[cols].rename(columns={
    'sku': 'SKU', 'product_name': 'Product',
    'avg_daily_demand': 'Avg Demand/day', 'safety_stock': 'Safety Stock',
    'reorder_point': 'Reorder Point', 'current_stock': 'Current Stock',
    'days_of_supply': 'Days Supply', 'status': 'Status'
}).set_index('SKU')

status_colors = {
    'OK':          'background-color: #d5f5e3',
    'REORDER NOW': 'background-color: #fef9e7',
    'CRITICAL':    'background-color: #fadbd8; font-weight: bold',
    'EXCESS':      'background-color: #e8daef',
    'STRANDED':    'background-color: #d5dbdb',
    'STOCKOUT':    'background-color: #c0392b; color: white; font-weight: bold',
}

inv_display.style\
    .format({'Avg Demand/day': '{:.1f}', 'Days Supply': '{:.1f}'})\
    .applymap(lambda v: status_colors.get(v, ''), subset=['Status'])

In [ ]:
fig = go.Figure()

colors = [STATUS_COLOR.get(s, '#95a5a6') for s in inv_df['status']]

fig.add_trace(go.Bar(
    x=inv_df['product_name'],
    y=inv_df['days_of_supply'],
    marker_color=colors,
    text=inv_df['status'],
    textposition='outside',
    name='Days of Supply'
))

fig.add_hline(y=90, line_dash='dash', line_color='#9b59b6',
              annotation_text='90d — Excess threshold (IPI penalty zone)')
fig.add_hline(y=30, line_dash='dot', line_color='#f39c12',
              annotation_text='30d — Reorder zone')

fig.update_layout(
    title='Days of Supply by SKU (status color-coded)',
    xaxis_title='Product',
    yaxis_title='Days of Supply',
    height=440,
    plot_bgcolor='white',
    showlegend=False
)
fig.show()

**Status breakdown:**
- **FC-002 (Yoga Mat) — EXCESS:** >90 days of supply. No immediate restock. Activate sell-through campaign.
- **FC-004 (Jump Rope) — CRITICAL:** <1 day of supply. Emergency air freight or mark as unavailable.
- **FC-008 (Lifting Belt) — REORDER NOW:** Stock has crossed the reorder point. Order from China supplier immediately (45-day LT).
- **FC-007 (Shaker) — STRANDED:** 200 units in FBA with no live listing. Revenue = €0/day. Fix listing first, then evaluate stock level.

---
## 5. IPI Score: Amazon’s Inventory Report Card

The **Inventory Performance Index (IPI)** is Amazon’s 0–1000 score that determines your FBA storage capacity.
Accounts below 400 face storage volume restrictions during peak periods (Q4, Prime Day).

Amazon does not publish the exact formula. This model uses a weighted approximation
based on publicly documented components and industry benchmarks:

| Component | Weight | Definition | Target |
|---|---|---|---|
| Sell-through rate | ~35% | Units sold 90d / avg inventory on hand | ≥ 3.0 |
| Excess inventory | ~35% | % of units with >90 days of supply | ≈ 0% |
| Stranded inventory | ~15% | % of units with no active listing | = 0% |
| In-stock rate | ~15% | % of days with stock available | ≥ 95% |

**Score thresholds:** <400 = penalized | 400–549 = at risk | 550–699 = good | 700+ = excellent

In [ ]:
total_units   = inv_df['current_stock'].sum()
excess_units  = inv_df[inv_df['is_excess']]['current_stock'].sum()
stranded_units = inv_df[inv_df['is_stranded']]['current_stock'].sum()

sell_through  = inv_df['sell_through_rate'].mean()
excess_pct    = (excess_units / total_units * 100) if total_units > 0 else 0
stranded_pct  = (stranded_units / total_units * 100) if total_units > 0 else 0
instock_rate  = inv_df[~inv_df['is_stranded']]['days_of_supply'].clip(0, 90).mean() / 90 * 100

score = estimate_ipi(sell_through, excess_pct, stranded_pct, instock_rate)
label, _ = ipi_label(score)

print("IPI Component Inputs:")
print(f"  Sell-through rate:  {sell_through:.2f}  (target ≥ 3.0)")
print(f"  Excess inventory:   {excess_pct:.1f}%  (target ≈ 0%)")
print(f"  Stranded inventory: {stranded_pct:.1f}%  (target = 0%)")
print(f"  In-stock rate:      {instock_rate:.1f}%  (target ≥ 95%)")
print(f"\n  Estimated IPI Score: {score} — {label}")

In [ ]:
fig = plot_gauge(score)
fig.show()

print("Recommendations:")
for tip in ipi_advice(sell_through, excess_pct, stranded_pct, instock_rate):
    print(f"  {tip}")

---
## 6. Campaign Readiness: Can We Launch the January Promo?

This is the **Ops × Growth coordination problem** — the most common operational breakdown in Amazon-first brands.
Growth wants to run a promotion. Ops needs to know: *do we have enough stock to survive it without stockout?*

**Scenario:**  
Growth proposes a **40% discount on FC-001 (Resistance Bands)** for 7 days.  
FC-001 is a Class A SKU — highest operational priority. Lead time: 35 days to China.

**Stock need calculation:**

$$Stock\_needed = \underbrace{avg\_demand \times (1 + uplift) \times campaign\_days}_{\text{campaign demand}} + \underbrace{avg\_demand \times lead\_time}_{\text{restock buffer}} + SS$$

If `current_stock ≥ stock_needed` → **Safe to launch**  
If stock runs out before the campaign ends → **Danger** — do not launch  
Otherwise → **Warning** — launch but order now

In [ ]:
result = check_readiness(
    inv_df=inv_df,
    sku='FC-001',
    uplift_pct=40,
    campaign_days=7
)

print("FC-001 — Resistance Bands Set")
print("Scenario: 40% discount × 7 days\n")
print(result['message'])
print(f"\nDays until stockout at campaign pace: {result['days_until_stockout']} days")
print(f"Order deadline (if needed): {result['order_deadline']}")

In [ ]:
breakdown = pd.DataFrame([
    {'Component': 'Campaign demand  (7d × avg×1.40)', 'Units': result['campaign_demand']},
    {'Component': 'Post-campaign restock buffer  (35d lead time)', 'Units': result['post_campaign_buffer']},
    {'Component': 'Safety stock  (95% service level)', 'Units': result['safety_stock']},
    {'Component': '── Total stock needed ──', 'Units': result['stock_needed']},
    {'Component': 'Current FBA stock', 'Units': result['current_stock']},
    {'Component': 'Units to order', 'Units': result['units_to_order']},
]).set_index('Component')

breakdown.style.format({'Units': '{:,.0f}'})\
    .applymap(lambda v: 'font-weight: bold' if v in [result['stock_needed'], result['units_to_order']] else '',
              subset=['Units'])

---
## 7. Conclusions & Action Plan

### Priority actions for FlexCore

| Priority | SKU | Action | Reason |
|---|---|---|---|
| **Immediate** | FC-004 Jump Rope | Emergency restock (air freight if needed) | CRITICAL — stockout today |
| **Immediate** | FC-007 Shaker | Fix stranded listing in Seller Central | 200 units generating €0/day |
| **This week** | FC-008 Lifting Belt | Place ocean freight order (45d LT) | Below ROP |
| **This week** | FC-002 Yoga Mat | Liquidation or promo to reduce excess | 100+ days supply — IPI drag |
| **28d before launch** | FC-001 Resistance Bands | Pre-order units for January campaign | Campaign readiness confirmed |

### Using this model with real data

Replace `simulate_data.py` with your actual exports. Two DataFrames required:

```python
# sku_df — product master
sku_df = pd.read_csv('products.csv')
# Required: sku, product_name, price, cogs, lead_time_days, current_stock, is_stranded

# demand_df — daily sales history (90+ days recommended)
demand_df = pd.read_csv('sales_history.csv')
# Required: date, sku, units_sold
```

**Amazon Seller Central sources:**
- `Reports → Sales & Traffic` — daily units ordered by ASIN
- `Inventory → FBA Inventory` — current stock levels, stranded flags
- `Restock Tool` — lead times per supplier (already in your catalog)

---

*Built by Roberto Galarza | Operations & AI | [linkedin.com/in/robertogalarza](https://linkedin.com/in/robertogalarza)*